Bootstrapping
Nombre: Juan Pablo Moran
Expediente: 753760

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("Advertising.csv")
df.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


Regresion de sales con TV, radio y newspaper usando OLS.

In [2]:
X = df[["TV", "radio", "newspaper"]]
X = sm.add_constant(X)
y = df["sales"]

ols = sm.OLS(y, X)
results = ols.fit()
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     570.3
Date:                Thu, 19 Nov 2026   Prob (F-statistic):           1.58e-96
Time:                        16:49:38   Log-Likelihood:                -386.18
No. Observations:                 200   AIC:                             780.4
Df Residuals:                     196   BIC:                             793.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9389      0.312      9.422      0.000       2.324       3.554
TV             0.0458      0.001     32.809      0.000       0.043       0.049
radio          0.1885      0.009     21.893      0.000       0.172       0.206
newspaper     -0.0010      0.006     -0.177      0.860      -0.013       0.011
==============================================================================
Omnibus:                       60.414   Durbin-Watson:                   2.084
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              151.241
Skew:                          -1.327   Prob(JB):                     1.44e-33
Kurtosis:                       6.332   Cond. No.                         454.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

TV y radio salen significativas, newspaper no por su p value alto. Lo que se gasta en periodico no se nota en las ventas una vez que ya se toma en cuenta TV y radioor cada 1,000 dolares mas en TV las ventas suben como 46 unidades, y en radio como 189.  Y con un R2 de 0.897 con estas variables se explica casi el 90% de las ventas

In [9]:
from sklearn.utils import resample
Xb, yb = resample(X, y, random_state= 5)
print(Xb, yb)

     const     TV  radio  newspaper
154    1.0  187.8   21.1        9.5
98     1.0  289.7   42.3       51.2
35     1.0  290.7    4.1        8.5
197    1.0  177.0    9.3        6.4
47     1.0  239.9   41.5       18.5
..     ...    ...    ...        ...
38     1.0   43.1   26.7       35.1
86     1.0   76.3   27.5       16.0
19     1.0  147.3   23.9       19.1
72     1.0   26.8   33.0       19.3
43     1.0  206.9    8.4       26.4

[200 rows x 4 columns] 154    15.6
98     25.4
35     12.8
197    12.8
47     23.2
       ... 
38     10.1
86     12.0
19     14.6
72      8.8
43     12.9
Name: sales, Length: 200, dtype: float64


In [ ]:
resample saca una muestra del mismo tamaño pero con reemplazo, entonces hay filas repetidas y otras que no salen.

In [10]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(Xb, yb)

print("b0 =", lr.intercept_)
print("b TV, radio, newspaper =", lr.coef_)
print("R2 =", lr.score(Xb, yb))

b0 = 3.429320066932
b TV, radio, newspaper = [ 0.          0.04234393  0.20226639 -0.00962437]
R2 = 0.9088847924656289


Los coeficientes salen parecidos a los de OLS pero no iguales, cambian un poco segun que filas salieron en la muestra y se quito la columna const porque LinearRegression ya saca el intercepto

In [20]:
Xb,yb = resample(X, y, random_state = 6)
X = df[["TV", "radio", "newspaper"]]
y = df["sales"]

b0 = []
b_tv = []
b_radio = []
b_news = []

for i in range(10000):
    Xb, yb = resample(X, y, random_state=i)
    lr = LinearRegression()
    lr.fit(Xb, yb)
    b0.append(lr.intercept_)
    b_tv.append(lr.coef_[0])
    b_radio.append(lr.coef_[1])
    b_news.append(lr.coef_[2])

print("B0: prom =", np.mean(b0), " S =", np.std(b0, ddof=1))
print("TV: prom =", np.mean(b_tv), " S =", np.std(b_tv, ddof=1))
print("Radio: prom =", np.mean(b_radio), " S =", np.std(b_radio, ddof=1))
print("Newspaper: prom =", np.mean(b_news), " S =", np.std(b_news, ddof=1))

B0: prom = 2.9534439182595076  S = 0.33735721110087824
TV: prom = 0.045695405232301985  S = 0.0019086612044736414
Radio: prom = 0.18836691153579455  S = 0.010860014834065238
Newspaper: prom = -0.0009026654029186444  S = 0.006480714244211998


Se repitio la regresion 10,000 veces con muestras distintas y se guardaron los coeficientes, el promedio sale casi igual que en OLS entonces los coeficientes son estables y no dependen de unos cuantos datos y la desviacion estandar de los coeficientes es el error estandar sin usar formulas.